# Train 5 classifiers (Logistic, DecisionTree, KNN, GaussianNB, RandomForest)

This notebook trains the required pipelines, exports `model/*.pkl`, `metrics.json`, `schema.json`, and `../test_data.csv`. Follow the assignment `CLAUDE_1.md`.

In [ ]:
# Imports and config
RANDOM_STATE = 42
import json
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
print('scikit-learn', sklearn.__version__)

In [ ]:
# Load and inspect
DATA_PATH = Path('..') / 'data' / 'students' / 'data.csv'
df = pd.read_csv(DATA_PATH, sep=None, engine='python')
print('shape:', df.shape)
print('dtypes:
', df.dtypes.value_counts())
print('missing per column:
', df.isna().sum().sort_values(ascending=False).head(10))
if 'Target' in df.columns:
    print('
Target distribution:
', df['Target'].value_counts())
else:
    raise SystemExit('Target column not found; aborting')

In [ ]:
# Column typing (detect nominal integer-coded columns)
FEATURE_COLUMNS = [c for c in df.columns if c != 'Target']
numeric_cols = [c for c in df.select_dtypes(include=['number']).columns if c != 'Target']
nominal_candidates = []
for c in numeric_cols:
    if pd.api.types.is_integer_dtype(df[c]) and df[c].nunique(dropna=True) <= 30:
        nominal_candidates.append(c)
NOMINAL_COLS = nominal_candidates
NUMERIC_COLS = [c for c in FEATURE_COLUMNS if c not in NOMINAL_COLS]
print('NUMERIC_COLS count:', len(NUMERIC_COLS))
print('NOMINAL_COLS count:', len(NOMINAL_COLS))
print('NOMINAL_EXAMPLE', NOMINAL_COLS[:20])

In [ ]:
# Split (stratified) and export raw test set
from sklearn.model_selection import train_test_split
X = df[FEATURE_COLUMNS].copy()
y = df['Target'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)
output_test = Path('..') / 'test_data.csv'
test_df.to_csv(output_test, index=False)
print('Wrote test_data.csv with shape', test_df.shape)

In [ ]:
# Preprocessor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
num_pipeline = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
cat_pipeline = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([(
    ('num', num_pipeline, NUMERIC_COLS),
    ('cat', cat_pipeline, NOMINAL_COLS),
)])
print('Preprocessor created')

In [ ]:
# Model definitions and training loop
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'decision_tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=RANDOM_STATE),
    'knn': KNeighborsClassifier(n_neighbors=15),
    'naive_bayes': GaussianNB(),
    'random_forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
}
Path('..' + '/model').mkdir(parents=True, exist_ok=True)
metrics = {}
def compute_metrics(y_true, y_pred, y_proba, classes):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro', labels=classes)
    return {'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1), 'mcc': float(mcc), 'auc': float(auc)}

for slug, clf in models.items():
    print('Training', slug)
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)
    m = compute_metrics(y_test, y_pred, y_proba, classes=pipe.classes_)
    metrics[slug] = {'display_name': slug.replace('_', ' ').title(), **m}
    joblib.dump(pipe, Path('..') / 'model' / f'{slug}.pkl')
    print('Saved', slug, '-> model/' + slug + '.pkl')

# Write metrics.json and schema.json
with open(Path('..') / 'model' / 'metrics.json', 'w', encoding='utf8') as f:
    json.dump(metrics, f, indent=2)
schema = {
    'target_column': 'Target',
    'feature_columns': FEATURE_COLUMNS,
    'class_labels': sorted(df['Target'].unique().tolist()),
    'nominal_columns': NOMINAL_COLS,
    'sklearn_version': sklearn.__version__,
    'n_train': int(X_train.shape[0]),
    'n_test': int(X_test.shape[0]),
}
with open(Path('..') / 'model' / 'schema.json', 'w', encoding='utf8') as f:
    json.dump(schema, f, indent=2)
print('Wrote metrics.json and schema.json')